# Convert Unsupported Types for Fabric SQL Analytics Endpoint
Reads `all_datatypes_demo` and creates a new Delta table with types converted for SQL Analytics Endpoint compatibility.

**Conversions:**
| Source Type | Target Type | Reason |
|---|---|---|
| TIMESTAMP_NTZ | TIMESTAMP | SQL endpoint doesn't support TIMESTAMP_NTZ |
| ARRAY | STRING (JSON) | SQL endpoint doesn't support complex ARRAY type |
| MAP | STRING (JSON) | SQL endpoint doesn't support complex MAP type |

In [ ]:
from pyspark.sql.functions import col, to_json, cast
from pyspark.sql.types import TimestampType

In [ ]:
# Read source table

SOURCE_TABLE = "all_datatypes_demo"
TARGET_TABLE = "all_datatypes_demo_sql_compatible"

df = spark.read.table(SOURCE_TABLE)

print("=== Source Schema ===")
df.printSchema()
print(f"Row count: {df.count()}")

In [ ]:
# Convert unsupported types for SQL Analytics Endpoint

df_converted = (
    df
    # TIMESTAMP_NTZ -> TIMESTAMP (cast preserves the datetime value, assigns UTC context)
    .withColumn("timestamp_ntz_col", col("timestamp_ntz_col").cast(TimestampType()))
    # ARRAY -> STRING (JSON representation)
    .withColumn("array_col", to_json(col("array_col")))
    # MAP -> STRING (JSON representation)
    .withColumn("map_col", to_json(col("map_col")))
)

print("=== Converted Schema ===")
df_converted.printSchema()

print("\n=== Sample converted data ===")
df_converted.select(
    "id", "timestamp_col", "timestamp_ntz_col", "array_col", "map_col"
).show(10, truncate=False)

In [ ]:
# Write converted table as Delta

df_converted.write.format("delta").mode("overwrite").saveAsTable(TARGET_TABLE)

print(f"Delta table '{TARGET_TABLE}' created successfully with {df_converted.count()} rows.")

In [ ]:
# Verify the new table schema and data

df_verify = spark.read.table(TARGET_TABLE)

print("=== Final Table Schema ===")
df_verify.printSchema()

print("\n=== DESCRIBE EXTENDED ===")
spark.sql(f"DESCRIBE EXTENDED {TARGET_TABLE}").show(100, truncate=False)

print("\n=== Comparison: timestamp_col vs timestamp_ntz_col (both now TIMESTAMP) ===")
df_verify.select("id", "timestamp_col", "timestamp_ntz_col").show(5, truncate=False)

print("\n=== Converted complex types (now STRING) ===")
df_verify.select("id", "array_col", "map_col").show(5, truncate=False)